# Kinase docking: ABL1 and imatinib

In this lab we dock kinase inhibitors into the kinase **ABL1** (PDB **1IEP**, ABL1 bound to imatinib), using the same tools as the Bioinformatics and Docking labs: **Biopython**, **LePro**, **RDKit**, **AutoDock Vina** (Python bindings), **py3Dmol** and **nglview**.

## Content of this notebook

1. Fetching the structure and splitting protein and ligand
2. Finding the kinase landmarks (hinge, gatekeeper, DFG) with the KLIFS pocket
3. Protein preparation with LePro and system visualization
4. Docking with AutoDock Vina
    - Receptor and ligand preparation (PDBQT)
    - Docking box definition
    - Re-docking imatinib and checking the pose (RMSD, hydrogen bonds)
    - 3D visualization of the docking results
5. Docking a panel of approved kinase inhibitors
6. The T315I gatekeeper mutation: why imatinib fails and ponatinib works
7. Analyzing the results

**Runtime:** re-docking takes 1–3 minutes; the panel of 9 inhibitors 10–20 minutes (set `EXHAUSTIVENESS = 4` to halve this).

In [ ]:
import os
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import py3Dmol
import nglview

from vina import Vina

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdMolAlign
from Bio.PDB import PDBParser, PDBIO, Select

import warnings
warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
%config Completer.use_jedi = False

As with the previous labs, we will first make a folder to work in:

In [ ]:
import sys, importlib, importlib.util

def find_course_dir(start):
    """The course folder is the one that contains scripts/viewer.py (and bin/lepro)."""
    for d in [start, *start.parents]:
        if (d / 'scripts' / 'viewer.py').exists():
            return d
    return None

# _dh[0] is the folder the notebook was started from; also try the current folder
COURSE_DIR = find_course_dir(Path(_dh[0]).resolve()) or find_course_dir(Path.cwd().resolve())
if COURSE_DIR is None:
    raise FileNotFoundError("Could not find scripts/viewer.py in this folder or any parent folder. "
                            "Put this notebook in the course folder (next to 'scripts' and 'bin').")
HOMEDIR = str(COURSE_DIR)

# import viewer from <course folder>/scripts, even if another 'scripts' package is on the path
if HOMEDIR in sys.path:
    sys.path.remove(HOMEDIR)
sys.path.insert(0, HOMEDIR)
for mod in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')]:
    del sys.modules[mod]
try:
    from scripts import viewer
except ImportError:                      # fall back to loading the file directly
    spec = importlib.util.spec_from_file_location('viewer', COURSE_DIR / 'scripts' / 'viewer.py')
    viewer = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(viewer)
print('course folder :', HOMEDIR)
print('viewer module :', viewer.__file__)

os.chdir(HOMEDIR)
# We need to check whether the directory is there
os.makedirs('KinaseDocking', exist_ok=True)
os.chdir('KinaseDocking')
print('working folder:', os.getcwd())

In [ ]:
PDBCODE = '1IEP'       # ABL1 kinase domain with imatinib
PDB_CHAIN = 'A'
LIGAND_CODE = 'STI'    # imatinib (STI-571)
IMATINIB = 'Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1'

EXHAUSTIVENESS = 4     # search thoroughness (the Docking lab used 10)
N_POSES = 5
SEED = 1234            # makes docking reproducible; change it to get different results

## 1. Fetching the structure and splitting protein and ligand

1IEP contains two copies of the ABL1–imatinib complex (chains A and B). We keep chain A, and save the protein and imatinib to separate files, just like in the Bioinformatics lab.

In [ ]:
pdb_file = f'{PDBCODE}.pdb'
if not os.path.exists(pdb_file):
    r = requests.get(f'https://files.rcsb.org/download/{PDBCODE}.pdb', timeout=60)
    r.raise_for_status()
    open(pdb_file, 'w').write(r.text)

class ProteinChain(Select):
    """Standard amino acids of one chain (no waters, no ligands)."""
    def accept_chain(self, chain):
        return chain.id == PDB_CHAIN
    def accept_residue(self, residue):
        return residue.id[0] == ' '
    def accept_atom(self, atom):
        return (not atom.is_disordered()) or atom.get_altloc() == 'A'

class LigandInChain(Select):
    """Only the ligand with residue name `code` in the selected chain."""
    def __init__(self, code):
        self.code = code
    def accept_chain(self, chain):
        return chain.id == PDB_CHAIN
    def accept_residue(self, residue):
        return residue.get_resname() == self.code

structure = PDBParser(QUIET=True).get_structure(PDBCODE, pdb_file)
io = PDBIO()
io.set_structure(structure)
io.save(f'protein-{PDBCODE}.pdb', ProteinChain())
io.save(f'ligand-{LIGAND_CODE}.pdb', LigandInChain(LIGAND_CODE))
print('saved', f'protein-{PDBCODE}.pdb', 'and', f'ligand-{LIGAND_CODE}.pdb')

PDB files contain no bond orders and no hydrogens. To compare docking poses with the crystal pose we need a proper RDKit molecule of the crystal ligand, so we assign the bond orders from the imatinib SMILES:

In [ ]:
xtal_lig = Chem.MolFromPDBFile(f'ligand-{LIGAND_CODE}.pdb', removeHs=True)
xtal_lig = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(IMATINIB), xtal_lig)
print('crystal ligand heavy atoms:', xtal_lig.GetNumAtoms())
xtal_lig

## 2. Where are the hinge, gatekeeper and DFG motif?

All kinases share the same ATP pocket. The **KLIFS** database (klifs.net) defines 85 pocket residues that are aligned across all kinases, so position 45 is always the **gatekeeper**, 46–48 the **hinge**, 17 the catalytic **β3-Lys**, 24 the **αC-Glu** and 81–83 the **DFG** motif. Below we align the ABL1 KLIFS pocket to the sequence of our structure to find these residues automatically — residue numbers differ between structures and papers, so this is safer than typing them in.

In [ ]:
ABL1_POCKET = "HKLGGGQYGEVYEVAVKTLEFLKEAAVMKEIKPNLVQLLGVYIITEFMTYGNLLDYLREYLEKKNFIHRDLAARNCLVVADFGLS"
THREE2ONE = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G',
             'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
             'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V', 'HID': 'H', 'HIE': 'H', 'HIP': 'H', 'CYX': 'C'}

def read_pdb_atoms(path):
    """ATOM/HETATM records as a table (element taken from the name if the element column is empty)."""
    rows = []
    for l in open(path):
        if l.startswith(('ATOM', 'HETATM')):
            name = l[12:16].strip()
            elem = l[76:78].strip()
            if not elem.isalpha():
                elem = ''.join(ch for ch in name if ch.isalpha())[:1]
            rows.append((name, l[17:20].strip(), l[21], int(l[22:26]),
                         float(l[30:38]), float(l[38:46]), float(l[46:54]), elem.capitalize()))
    return pd.DataFrame(rows, columns=['name', 'resname', 'chain', 'resid', 'x', 'y', 'z', 'element'])

def align_pocket(seq, pocket, match=5, mismatch=-4, gap_open=-6, gap_extend=0, missing=-6):
    """Align the KLIFS pocket to a full sequence (local dynamic programming with affine gaps).
    Returns {KLIFS position: index in seq}."""
    q = [(i + 1, aa) for i, aa in enumerate(pocket) if aa != '-']
    m, n = len(q), len(seq)
    NEG = float('-inf')
    M = [[NEG] * (n + 1) for _ in range(m + 1)]
    X = [[NEG] * (n + 1) for _ in range(m + 1)]
    Y = [[NEG] * (n + 1) for _ in range(m + 1)]
    for j in range(n + 1):
        M[0][j] = 0.0
    for i in range(1, m + 1):
        X[i][0] = missing * i
        for j in range(1, n + 1):
            s = match if q[i - 1][1] == seq[j - 1] else mismatch
            M[i][j] = max(M[i-1][j-1], X[i-1][j-1], Y[i-1][j-1]) + s
            X[i][j] = max(M[i-1][j], X[i-1][j], Y[i-1][j]) + missing
            Y[i][j] = max(M[i][j-1] + gap_open, X[i][j-1] + gap_open, Y[i][j-1] + gap_extend)
    j = max(range(n + 1), key=lambda jj: max(M[m][jj], X[m][jj]))
    i, state = m, ('M' if M[m][j] >= X[m][j] else 'X')
    mapping = {}
    while i > 0:
        if state == 'M':
            mapping[q[i - 1][0]] = j - 1
            prev = {'M': M[i-1][j-1], 'X': X[i-1][j-1], 'Y': Y[i-1][j-1]}
            i, j = i - 1, j - 1
        elif state == 'X':
            prev = {'M': M[i-1][j], 'X': X[i-1][j], 'Y': Y[i-1][j]}
            i -= 1
        else:
            prev = {'M': M[i][j-1] + gap_open, 'X': X[i][j-1] + gap_open, 'Y': Y[i][j-1] + gap_extend}
            j -= 1
        if i == 0:
            break
        state = max(prev, key=prev.get)
    return mapping

atoms = read_pdb_atoms(f'protein-{PDBCODE}.pdb')
ca = atoms[atoms.name == 'CA'].reset_index(drop=True)
seq = ''.join(THREE2ONE.get(r, 'X') for r in ca.resname)
kmap = {pos: (ca.resname[i], int(ca.resid[i])) for pos, i in align_pocket(seq, ABL1_POCKET).items()}

LANDMARKS = {'β3-Lys': 17, 'αC-Glu': 24, 'gatekeeper': 45, 'hinge 1': 46, 'hinge 2': 47,
             'hinge Met': 48, 'DFG-Asp': 81, 'DFG-Phe': 82}
landmarks = pd.DataFrame([(k, p, *kmap[p]) for k, p in LANDMARKS.items()],
                         columns=['landmark', 'KLIFS position', 'residue', 'residue number'])
RES = dict(zip(landmarks.landmark, landmarks['residue number']))
landmarks

You should find K271, E286, **T315** (gatekeeper), E316–F317–**M318** (hinge) and D381–F382. These are the residues described in every imatinib paper. Remember them, we will look for interactions with them later.

## 3. Protein preparation with LePro and system visualization

As in the Bioinformatics lab, we use **LePro** (part of LeDock) to add hydrogen atoms to the protein.

In [ ]:
LEPRO = f'{HOMEDIR}/bin/lepro'

def lepro(pdb_in, pdb_out):
    if not os.path.exists(LEPRO):
        raise FileNotFoundError(f'LePro not found at {LEPRO}')
    os.system(f'"{LEPRO}" {pdb_in}')
    shutil.move('pro.pdb', pdb_out)
    return pdb_out

lepro(f'protein-{PDBCODE}.pdb', f'{PDBCODE}_prepped.pdb')
prepped = read_pdb_atoms(f'{PDBCODE}_prepped.pdb')
print(len(prepped), 'protein atoms after adding hydrogens')
# the landmark residue numbers must still be valid in the prepared file
if sorted(prepped[prepped.name == 'CA'].resid) != sorted(ca.resid):
    print('WARNING: residue numbering changed during preparation -> re-mapping the KLIFS landmarks')
    ca2 = prepped[prepped.name == 'CA'].reset_index(drop=True)
    seq2 = ''.join(THREE2ONE.get(r, 'X') for r in ca2.resname)
    kmap2 = {pos: int(ca2.resid[i]) for pos, i in align_pocket(seq2, ABL1_POCKET).items()}
    RES = {k: kmap2[p] for k, p in LANDMARKS.items()}
print(RES)

In [ ]:
view = py3Dmol.view(width=800, height=500)
view.addModel(open(f'{PDBCODE}_prepped.pdb').read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white'}})
colours = {'gatekeeper': 'red', 'hinge Met': 'orange', 'β3-Lys': 'blue', 'αC-Glu': 'blue',
           'DFG-Asp': 'magenta', 'DFG-Phe': 'magenta'}
for name, colour in colours.items():
    view.addStyle({'model': 0, 'resi': RES[name]}, {'stick': {'color': colour, 'radius': 0.2}})
view.addModel(open(f'ligand-{LIGAND_CODE}.pdb').read(), 'pdb')
view.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.25}})
view.zoomTo({'model': 1})
print('imatinib: green | gatekeeper T315: red | hinge M318: orange | K271/E286: blue | DFG: magenta')
view.show()

Imatinib binds the **inactive "DFG-out"** conformation of ABL1: the DFG-Phe (magenta) has flipped, which opens a pocket behind the gatekeeper where imatinib's benzamide–piperazine tail binds. Drugs that bind this way are called **type II** kinase inhibitors.

## 4. Docking with AutoDock Vina

### 4.1 Receptor and ligand preparation (PDBQT)

Vina needs PDBQT files: atom coordinates plus an AutoDock atom type (`C`/`A` aliphatic/aromatic carbon, `N`/`NA` nitrogen/acceptor nitrogen, `OA`, `SA`, `HD` polar hydrogen).

* **Ligands** are converted with the same `MolToPDBQTBlock` function as in the Docking lab. It also defines the rotatable bonds (the torsion tree) and adds hydrogens on H-bond donors only.
* **Receptor:** Vina needs the polar hydrogens of the protein (`HD`) to recognise H-bond donors, such as the backbone NH of the hinge Met318 that binds every kinase inhibitor. `Chem.MolFromPDBFile` removes hydrogens by default, so for the protein we use a small converter that keeps the polar hydrogens LePro added and drops the non-polar ones.

In [ ]:
from math import isnan, isinf

def PDBQTAtomLines(mol, donors, acceptors):
    """Create a list with PDBQT atom lines for each atom in the molecule. Donors
    and acceptors are given as a list of atom indices.
    """

    atom_lines = [line.replace('HETATM', 'ATOM  ')
                  for line in Chem.MolToPDBBlock(mol).split('\n')
                  if line.startswith('HETATM') or line.startswith('ATOM')]

    pdbqt_lines = []
    for idx, atom in enumerate(mol.GetAtoms()):
        pdbqt_line = atom_lines[idx][:56]

        pdbqt_line += '0.00  0.00    '  # append empty vdW and ele
        # Get charge
        charge = 0.
        fields = ['_MMFF94Charge', '_GasteigerCharge', '_TriposPartialCharge']
        for f in fields:
            if atom.HasProp(f):
                charge = atom.GetDoubleProp(f)
                break
        # FIXME: this should not happen, blame RDKit
        if isnan(charge) or isinf(charge):
            charge = 0.
        pdbqt_line += ('%.3f' % charge).rjust(6)

        # Get atom type
        pdbqt_line += ' '
        atomicnum = atom.GetAtomicNum()
        if atomicnum == 6 and atom.GetIsAromatic():
            pdbqt_line += 'A'
        elif atomicnum == 7 and idx in acceptors:
            pdbqt_line += 'NA'
        elif atomicnum == 8 and idx in acceptors:
            pdbqt_line += 'OA'
        elif atomicnum == 1 and atom.GetNeighbors()[0].GetIdx() in donors:
            pdbqt_line += 'HD'
        else:
            pdbqt_line += atom.GetSymbol()
        pdbqt_lines.append(pdbqt_line)
    return pdbqt_lines

def MolToPDBQTBlock(mol, flexible=True, addHs=False, computeCharges=False):
    """Write RDKit Molecule to a PDBQT block

    Parameters
    ----------
        mol: rdkit.Chem.rdchem.Mol
            Molecule with a protein ligand complex
        flexible: bool (default=True)
            Should the molecule encode torsions. Ligands should be flexible,
            proteins in turn can be rigid.
        addHs: bool (default=False)
            The PDBQT format requires at least polar Hs on donors. By default Hs
            are added.
        computeCharges: bool (default=False)
            Should the partial charges be automatically computed. If the Hs are
            added the charges must and will be recomputed. If there are no
            partial charge information, they are set to 0.0.

    Returns
    -------
        block: str
            String wit PDBQT encoded molecule
    """
    # make a copy of molecule
    mol = Chem.Mol(mol)

    # if flexible molecule contains multiple fragments write them separately
    if flexible and len(Chem.GetMolFrags(mol)) > 1:
        return ''.join(MolToPDBQTBlock(frag, flexible=flexible, addHs=addHs,
                                       computeCharges=computeCharges)
                       for frag in Chem.GetMolFrags(mol, asMols=True))

    # Identify donors and acceptors for atom typing
    # Acceptors
    patt = Chem.MolFromSmarts('[$([O;H1;v2]),'
                              '$([O;H0;v2;!$(O=N-*),'
                              '$([O;-;!$(*-N=O)]),'
                              '$([o;+0])]),'
                              '$([n;+0;!X3;!$([n;H1](cc)cc),'
                              '$([$([N;H0]#[C&v4])]),'
                              '$([N&v3;H0;$(Nc)])]),'
                              '$([F;$(F-[#6]);!$(FC[F,Cl,Br,I])])]')
    acceptors = list(map(lambda x: x[0],
                         mol.GetSubstructMatches(patt, maxMatches=mol.GetNumAtoms())))
    # Donors
    patt = Chem.MolFromSmarts('[$([N&!H0&v3,N&!H0&+1&v4,n&H1&+0,$([$([Nv3](-C)(-C)-C)]),'
                              '$([$(n[n;H1]),'
                              '$(nc[n;H1])])]),'
                              # Guanidine can be tautormeic - e.g. Arginine
                              '$([NX3,NX2]([!O,!S])!@C(!@[NX3,NX2]([!O,!S]))!@[NX3,NX2]([!O,!S])),'
                              '$([O,S;H1;+0])]')
    donors = list(map(lambda x: x[0],
                      mol.GetSubstructMatches(patt, maxMatches=mol.GetNumAtoms())))
    if addHs:
        mol = Chem.AddHs(mol, addCoords=True, onlyOnAtoms=donors, )
    if addHs or computeCharges:
        AllChem.ComputeGasteigerCharges(mol)

    atom_lines = PDBQTAtomLines(mol, donors, acceptors)
    assert len(atom_lines) == mol.GetNumAtoms()

    pdbqt_lines = []

    # vina scores
    if (mol.HasProp('vina_affinity') and mol.HasProp('vina_rmsd_lb') and
            mol.HasProp('vina_rmsd_lb')):
        pdbqt_lines.append('REMARK VINA RESULT:  ' +
                           ('%.1f' % float(mol.GetProp('vina_affinity'))).rjust(8) +
                           ('%.3f' % float(mol.GetProp('vina_rmsd_lb'))).rjust(11) +
                           ('%.3f' % float(mol.GetProp('vina_rmsd_ub'))).rjust(11))

    pdbqt_lines.append('REMARK  Name = ' +
                       (mol.GetProp('_Name') if mol.HasProp('_Name') else ''))
    if flexible:
        # Find rotatable bonds
        rot_bond = Chem.MolFromSmarts('[!$(*#*)&!D1&!$(C(F)(F)F)&'
                                      '!$(C(Cl)(Cl)Cl)&'
                                      '!$(C(Br)(Br)Br)&'
                                      '!$(C([CH3])([CH3])[CH3])&'
                                      '!$([CD3](=[N,O,S])-!@[#7,O,S!D1])&'
                                      '!$([#7,O,S!D1]-!@[CD3]=[N,O,S])&'
                                      '!$([CD3](=[N+])-!@[#7!D1])&'
                                      '!$([#7!D1]-!@[CD3]=[N+])]-!@[!$(*#*)&'
                                      '!D1&!$(C(F)(F)F)&'
                                      '!$(C(Cl)(Cl)Cl)&'
                                      '!$(C(Br)(Br)Br)&'
                                      '!$(C([CH3])([CH3])[CH3])]')
        bond_atoms = list(mol.GetSubstructMatches(rot_bond))
        num_torsions = len(bond_atoms)

        # Active torsions header
        pdbqt_lines.append('REMARK  %i active torsions:' % num_torsions)
        pdbqt_lines.append('REMARK  status: (\'A\' for Active; \'I\' for Inactive)')
        for i, (a1, a2) in enumerate(bond_atoms):
            pdbqt_lines.append('REMARK%5.0i  A    between atoms: _%i  and  _%i'
                               % (i + 1, a1 + 1, a2 + 1))

        # Fragment molecule on bonds to ge rigid fragments
        bond_ids = [mol.GetBondBetweenAtoms(a1, a2).GetIdx()
                    for a1, a2 in bond_atoms]
        if bond_ids:
            mol_rigid_frags = Chem.FragmentOnBonds(mol, bond_ids, addDummies=False)
        else:
            mol_rigid_frags = mol
        frags = list(Chem.GetMolFrags(mol_rigid_frags))

        def weigh_frags(frag):
            """sort by the fragment size and the number of bonds (secondary)"""
            num_bonds = 0
            # bond_weight = 0
            for a1, a2 in bond_atoms:
                if a1 in frag or a2 in frag:
                    num_bonds += 1
                    # for frag2 in frags:
                    #     if a1 in frag2 or a2 in frag2:
                    #         bond_weight += len(frag2)

            # changed signs are fixing mixed sorting type (ascending/descending)
            return -len(frag), -num_bonds,  # bond_weight
        frags = sorted(frags, key=weigh_frags)

        # Start writting the lines with ROOT
        pdbqt_lines.append('ROOT')
        frag = frags.pop(0)
        for idx in frag:
            pdbqt_lines.append(atom_lines[idx])
        pdbqt_lines.append('ENDROOT')

        # Now build the tree of torsions usign DFS algorithm. Keep track of last
        # route with following variables to move down the tree and close branches
        branch_queue = []
        current_root = frag
        old_roots = [frag]

        visited_frags = []
        visited_bonds = []
        while len(frags) > len(visited_frags):
            end_branch = True
            for frag_num, frag in enumerate(frags):
                for bond_num, (a1, a2) in enumerate(bond_atoms):
                    if (frag_num not in visited_frags and
                        bond_num not in visited_bonds and
                        (a1 in current_root and a2 in frag or
                         a2 in current_root and a1 in frag)):
                        # direction of bonds is important
                        if a1 in current_root:
                            bond_dir = '%i %i' % (a1 + 1, a2 + 1)
                        else:
                            bond_dir = '%i %i' % (a2 + 1, a1 + 1)
                        pdbqt_lines.append('BRANCH %s' % bond_dir)
                        for idx in frag:
                            pdbqt_lines.append(atom_lines[idx])
                        branch_queue.append('ENDBRANCH %s' % bond_dir)

                        # Overwrite current root and stash previous one in queue
                        old_roots.append(current_root)
                        current_root = frag

                        # remove used elements from stack
                        visited_frags.append(frag_num)
                        visited_bonds.append(bond_num)

                        # mark that we dont want to end branch yet
                        end_branch = False
                        break
                    else:
                        continue
                    break  # break the outer loop as well

            if end_branch:
                pdbqt_lines.append(branch_queue.pop())
                if old_roots:
                    current_root = old_roots.pop()
        # close opened branches if any is open
        while len(branch_queue):
            pdbqt_lines.append(branch_queue.pop())
        pdbqt_lines.append('TORSDOF %i' % num_torsions)
    else:
        pdbqt_lines.extend(atom_lines)

    return '\n'.join(pdbqt_lines)

def MolFromPDBQTBlock(filename, sanitize=True, removeHs=True):
    """Read PDBQT block to a RDKit Molecule
    Parameters
    ----------
        block: string
            Residue name which explicitly pint to a ligand(s).
        sanitize: bool (default=True)
            Should the sanitization be performed
        removeHs: bool (default=True)
            Should hydrogens be removed when reading molecule.
    Returns
    -------
        mol: rdkit.Chem.rdchem.Mol
            Molecule read from PDBQT
    """
    pdb_lines = []
    name = ''
    data = {}
    with open(filename) as file:
        lines = file.readlines()
        block = [line.rstrip() for line in lines]

    for line in block: #.split('\n'):
        # Get all know data from REMARK section
        if line[:12] == 'REMARK  Name':
            name = line[15:].strip()
        elif line[:18] == 'REMARK VINA RESULT':
            tmp = line[19:].split()
            data['vina_affinity'] = tmp[0]
            data['vina_rmsd_lb'] = tmp[1]
            data['vina_rmsd_ub'] = tmp[2]

        # no more data to collect
        if line[:4] != 'ATOM':
            continue

        pdb_line = line[:56]
        pdb_line += '1.00  0.00           '

        # Do proper atom type lookup
        atom_type = line[71:].split()[1]
        if atom_type == 'A':
            atom_type = 'C'
        elif atom_type[:1] == 'O':
            atom_type = 'O'
        elif atom_type[:1] == 'H':
            atom_type = 'H'
            if removeHs:
                continue
        elif atom_type == 'NA':
            atom_type = 'N'

        pdb_lines.append(pdb_line + atom_type)
    mol = Chem.MolFromPDBBlock('\n'.join(pdb_lines), sanitize=False)
    if sanitize:
        Chem.SanitizeMol(mol)
    else:
        Chem.GetSSSR(mol)
    # reorder atoms using serial
    new_order = sorted(range(mol.GetNumAtoms()),
                       key=lambda i: (mol.GetAtomWithIdx(i)
                                      .GetPDBResidueInfo()
                                      .GetSerialNumber()))
    mol = Chem.RenumberAtoms(mol, new_order)

    # properties must be set on final copy of Mol, RenumberAtoms purges data
    mol.SetProp('_Name', name)
    for k, v in data.items():
        mol.SetProp(str(k), str(v))

    return mol

In [ ]:
AROMATIC_ATOMS = {
    'PHE': {'CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ'},
    'TYR': {'CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ'},
    'TRP': {'CG', 'CD1', 'CD2', 'NE1', 'CE2', 'CE3', 'CZ2', 'CZ3', 'CH2'},
    'HIS': {'CG', 'ND1', 'CD2', 'CE1', 'NE2'},
}
for h in ('HID', 'HIE', 'HIP'):
    AROMATIC_ATOMS[h] = AROMATIC_ATOMS['HIS']

def receptor_to_pdbqt(pdb_in, pdbqt_out):
    """Protein PDB with hydrogens -> rigid receptor PDBQT (polar H kept as HD, non-polar H dropped)."""
    lines = [l for l in open(pdb_in) if l.startswith(('ATOM', 'HETATM'))]
    df = read_pdb_atoms(pdb_in)
    xyz = df[['x', 'y', 'z']].values
    is_h = (df.element == 'H').values
    heavy = np.where(~is_h)[0]
    bonded_to = {i: heavy[np.argmin(np.linalg.norm(xyz[heavy] - xyz[i], axis=1))] for i in np.where(is_h)[0]}
    has_h = set(bonded_to.values())
    out = []
    for i, l in enumerate(lines):
        el, name, resn = df.element[i], df.name[i], df.resname[i]
        if el == 'H':
            if df.element[bonded_to[i]] not in ('N', 'O'):
                continue
            ad = 'HD'
        elif el == 'C':
            ad = 'A' if name in AROMATIC_ATOMS.get(resn, ()) else 'C'
        elif el == 'N':
            ad = 'NA' if (resn in ('HIS', 'HID', 'HIE', 'HIP') and name in ('ND1', 'NE2') and i not in has_h) else 'N'
        elif el == 'O':
            ad = 'OA'
        elif el == 'S':
            ad = 'SA'
        else:
            ad = el
        out.append(f'{l[:54]}{1.0:6.2f}{0.0:6.2f}    {0.0:+6.3f} {ad:<2s}')
    open(pdbqt_out, 'w').write('\n'.join(out) + '\n')
    return pdbqt_out

receptor_to_pdbqt(f'{PDBCODE}_prepped.pdb', 'protein.pdbqt')
print(open('protein.pdbqt').readline())

In [ ]:
def prepare_ligand(smiles, name, seed=SEED):
    """SMILES -> 3D conformer (ETKDG + MMFF) -> PDBQT file. Returns (heavy-atom mol, pdbqt text)."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise ValueError(f'could not generate 3D coordinates for {name}')
    AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
    heavy = Chem.RemoveHs(mol)                          # MolToPDBQTBlock adds back the polar H
    heavy.SetProp('_Name', name)
    block = MolToPDBQTBlock(heavy, flexible=True, addHs=True, computeCharges=True)
    open(f'{name}.pdbqt', 'w').write(block)
    Chem.MolToMolFile(heavy, f'{name}.mol')
    return heavy, block

ref_mol, ref_block = prepare_ligand(IMATINIB, 'ligand_00')
print('\n'.join(ref_block.splitlines()[:8]))

Note that the ligand now has real 3D coordinates (from RDKit's ETKDG conformer generator). Vina changes the torsions of the ligand during docking, but not its bond lengths, bond angles or ring shapes, so a good starting geometry matters.

### 4.2 Box definition

As in the Docking lab, the box is centred on the crystal ligand and its size is based on the ligand's radius of gyration (Feinstein & Brylinski, J Cheminform 2015).

In [ ]:
lig = read_pdb_atoms(f'ligand-{LIGAND_CODE}.pdb')
lig_xyz = lig[['x', 'y', 'z']].values
CenterOfGeometry = lig_xyz.mean(axis=0).round(3).tolist()
Rg = float(np.sqrt(((lig_xyz - lig_xyz.mean(axis=0)) ** 2).sum(axis=1).mean()))
BOX = round(max(Rg * 2.9, 20.0), 1)       # imatinib is long, so use at least 20 Å
print(f'Center Of Geometry = {CenterOfGeometry}')
print(f'Radius of Gyration = {Rg:.2f} Å  ->  box edge {BOX} Å')

### 4.3 Re-docking imatinib

We first check whether Vina can reproduce the crystal pose (**re-docking**). A pose within **2 Å RMSD** of the crystal pose is normally considered correct.

***This step takes a while!***

In [ ]:
def make_vina(receptor_pdbqt):
    v = Vina(sf_name='vina', seed=SEED, verbosity=0)
    v.set_receptor(receptor_pdbqt)
    v.compute_vina_maps(center=CenterOfGeometry, box_size=[BOX, BOX, BOX])
    return v

def split_models(pdbqt_text):
    models, cur = [], []
    for line in pdbqt_text.splitlines():
        if line.startswith('MODEL'):
            cur = []
        elif line.startswith('ENDMDL'):
            models.append('\n'.join(cur))
        else:
            cur.append(line)
    return models or [pdbqt_text]

def poses_to_mol(pdbqt_text, template, input_pdbqt):
    """Docked poses -> copy of `template` (heavy-atom RDKit mol) with one conformer per pose.

    The coordinates are matched by atom serial number: in the ligand PDBQT, heavy atom i of the
    template has serial number i+1 (hydrogens come after the heavy atoms), and Vina keeps these
    serial numbers in its output. We do not re-derive elements from the PDBQT atom types, because
    that is unreliable for two-letter elements such as Cl and Br."""
    n = template.GetNumAtoms()
    names_in = {int(l[6:11]): l[12:16] for l in input_pdbqt.splitlines()
                if l.startswith(('ATOM', 'HETATM')) and int(l[6:11]) <= n}
    out = Chem.Mol(template)
    out.RemoveAllConformers()
    for model in split_models(pdbqt_text):
        conf = Chem.Conformer(n)
        found = set()
        for l in model.splitlines():
            if not l.startswith(('ATOM', 'HETATM')):
                continue
            serial = int(l[6:11])
            if serial > n:                    # polar hydrogens
                continue
            if names_in.get(serial) != l[12:16]:
                raise ValueError(f'atom {serial} changed name ({names_in.get(serial)!r} -> {l[12:16]!r}); '
                                 'cannot map the docked pose onto the ligand')
            conf.SetAtomPosition(serial - 1, (float(l[30:38]), float(l[38:46]), float(l[46:54])))
            found.add(serial)
        if len(found) != n:
            raise ValueError(f'pose contains {len(found)} of {n} heavy atoms')
        out.AddConformer(conf, assignId=True)
    return out

def dock(v, smiles, name, exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES):
    mol, block = prepare_ligand(smiles, name)
    v.set_ligand_from_string(block)
    v.dock(exhaustiveness=exhaustiveness, n_poses=20)
    text = v.poses(n_poses=n_poses)
    open(f'{name}_docked.pdbqt', 'w').write(text)
    energies = v.energies(n_poses=n_poses)[:, 0]
    poses = poses_to_mol(text, mol, block)
    Chem.MolToPDBFile(poses, f'{name}_docked.pdb', confId=0)          # best pose, residue name UNL
    return poses, energies

def affinity_to_pki(dg):
    """Vina score (kcal/mol) -> approximate pKi, as in the Docking lab."""
    return -math.log10(math.exp(dg / 0.592))

v_wt = make_vina('protein.pdbqt')
redock_poses, redock_E = dock(v_wt, IMATINIB, 'ligand_00')
redock_rmsd = [rdMolAlign.CalcRMS(redock_poses, xtal_lig, prbId=c.GetId()) for c in redock_poses.GetConformers()]
pd.DataFrame({'pose': range(1, len(redock_E) + 1),
              'Vina score (kcal/mol)': redock_E.round(2),
              '≈ pKi': [round(affinity_to_pki(e), 2) for e in redock_E],
              'RMSD to crystal (Å)': np.round(redock_rmsd, 2)})

### 4.4 3D visualization of the docking results

In [ ]:
def show_pose(receptor_pdb, docked_pdb, ref_pdb=f'ligand-{LIGAND_CODE}.pdb'):
    view = py3Dmol.view(width=800, height=500)
    view.addModel(open(receptor_pdb).read(), 'pdb')
    view.setStyle({'model': 0}, {'cartoon': {'color': 'white'}})
    view.addStyle({'model': 0, 'resi': [RES[k] for k in colours]}, {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.15}})
    view.addModel(open(ref_pdb).read(), 'pdb')
    view.setStyle({'model': 1}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.2}})
    view.addModel(open(docked_pdb).read(), 'pdb')
    view.setStyle({'model': 2}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}})
    view.zoomTo({'model': 1})
    print('Reference: Magenta | Vina Pose: Cyan')
    view.show()

show_pose(f'{PDBCODE}_prepped.pdb', 'ligand_00_docked.pdb')

**If the molecule in cyan appears outside of the protein, please call your TA!**

### 4.5 Which interactions does the pose make?

Imatinib makes four characteristic hydrogen bonds in ABL1:

* **hinge Met318** backbone NH → pyridine N
* **gatekeeper Thr315** side-chain OH ↔ aniline NH
* **αC-Glu286** carboxylate ↔ amide NH
* **DFG-Asp381** backbone NH → amide C=O

We list the closest N/O–N/O distance between the ligand and each of these residues (≤ 3.5 Å counts as a hydrogen bond).

In [ ]:
def polar_contacts(mol, conf_id, receptor_pdb, residues, cutoff=3.5):
    rec = read_pdb_atoms(receptor_pdb)
    pos = mol.GetConformer(conf_id).GetPositions()
    lig_polar = [a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() in ('N', 'O')]
    rows = []
    for resid in residues:
        ra = rec[(rec.resid == resid) & rec.element.isin(['N', 'O'])]
        d = np.linalg.norm(ra[['x', 'y', 'z']].values[:, None, :] - pos[lig_polar][None], axis=-1)
        i, j = np.unravel_index(np.argmin(d), d.shape)
        rows.append({'residue': f'{ra.resname.iloc[i]}{resid}', 'protein atom': ra.name.iloc[i],
                     'ligand atom': f'{mol.GetAtomWithIdx(lig_polar[j]).GetSymbol()}{lig_polar[j]}',
                     'distance (Å)': round(float(d[i, j]), 2), 'H-bond?': bool(d[i, j] <= cutoff)})
    return pd.DataFrame(rows)

HB_RESIDUES = [RES['hinge Met'], RES['gatekeeper'], RES['αC-Glu'], RES['DFG-Asp']]
print('Crystal pose')
display(polar_contacts(xtal_lig, -1, f'{PDBCODE}_prepped.pdb', HB_RESIDUES))
print('Best docked pose')
display(polar_contacts(redock_poses, 0, f'{PDBCODE}_prepped.pdb', HB_RESIDUES))

We can also show the docked pose with its interactions in nglview, as in the Docking lab:

In [ ]:
# combine protein and ligand files
filenames = [f'{PDBCODE}_prepped.pdb', 'ligand_00_docked.pdb']
with open(f'{PDBCODE}-complex.pdb', 'w') as outfile:
    for fname in filenames:
        with open(fname) as infile:
            for line in infile:
                if not "END" in line:
                    outfile.write(line)

with open(f'{PDBCODE}-complex.pdb') as f:
    view = nglview.show_file(f, ext="pdb")

view.center("UNL")
viewer.show_residues_around(view, selection='ligand', radius=5.0)
surrounding_residues = viewer.get_residues_around(f'{PDBCODE}-complex.pdb', selection='UNL', radius=5.0)
view.add_contact(selection=f'ligand or {" or ".join(surrounding_residues.resid.astype(str))}',
                 hydrogen_bond=True)
view

Try to identify which of the following interactions between the docked molecule and the protein are displayed by hovering them in the viewer above (they are depicted as dashed lines).
1. hydrophobic interaction
2. salt bridge
3. hydrogen bond
4. π-π stacking
5. π-cation
6. halogen bond

**Question:** Re-run the docking with a different `SEED` and with `EXHAUSTIVENESS = 2` and `32`. Is the top pose always within 2 Å? How does the run time change?

## 5. Docking a panel of approved kinase inhibitors

We now dock nine approved drugs into the same ABL1 structure:

* **ABL1 inhibitors:** imatinib, nilotinib, ponatinib (type II, bind DFG-out like here); dasatinib, bosutinib (type I, bind the active DFG-in form);
* **not ABL1 inhibitors:** erlotinib and gefitinib (EGFR), palbociclib (CDK4/6);
* **asciminib:** a potent ABL1 inhibitor that binds a completely different site (the myristoyl pocket), not the ATP site.

Do the docking scores separate the ABL1 inhibitors from the others? What does Vina do with asciminib?

***This step takes 10–20 minutes.***

In [ ]:
PANEL = {
    'imatinib':    ('Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1', 'ABL1 (type II)'),
    'nilotinib':   ('Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc(C(F)(F)F)c2)cn1', 'ABL1 (type II)'),
    'ponatinib':   ('Cc1ccc(C(=O)Nc2ccc(CN3CCN(C)CC3)c(C(F)(F)F)c2)cc1C#Cc1cnc2cccnn12', 'ABL1 (type II)'),
    'dasatinib':   ('Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCO)CC2)n1', 'ABL1 (type I)'),
    'bosutinib':   ('COc1cc(Nc2c(C#N)cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)c(Cl)cc1Cl', 'ABL1 (type I)'),
    'asciminib':   ('O[C@@H]1CCN(c2ncc(C(=O)Nc3ccc(OC(F)(Cl)F)cc3)cc2-c2ccn[nH]2)C1', 'ABL1 (allosteric)'),
    'erlotinib':   ('COCCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC', 'not ABL1'),
    'gefitinib':   ('COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1', 'not ABL1'),
    'palbociclib': ('CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n(C2CCCC2)c1=O', 'not ABL1'),
}
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s, _ in PANEL.values()], molsPerRow=5,
                     subImgSize=(220, 180), legends=list(PANEL))

In [ ]:
import time
rows, panel_poses = [], {}
for name, (smi, label) in PANEL.items():
    t0 = time.time()
    poses, E = dock(v_wt, smi, name)
    panel_poses[name] = poses
    hac = poses.GetNumAtoms()
    rows.append({'ligand': name, 'class': label, 'Vina score': round(E[0], 2), '≈ pKi': round(affinity_to_pki(E[0]), 2),
                 'heavy atoms': hac, 'ligand efficiency': round(-E[0] / hac, 3),
                 'hinge H-bond?': bool(polar_contacts(poses, 0, f'{PDBCODE}_prepped.pdb', [RES['hinge Met']])['H-bond?'][0])})
    print(f'{name:12s} {E[0]:6.2f} kcal/mol  ({time.time() - t0:.0f} s)')
panel = pd.DataFrame(rows).sort_values('Vina score')
panel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {'ABL1 (type II)': 'tab:red', 'ABL1 (type I)': 'tab:orange', 'ABL1 (allosteric)': 'tab:purple', 'not ABL1': 'tab:grey'}
axes[0].barh(panel.ligand, panel['Vina score'], color=[colors[c] for c in panel['class']])
axes[0].invert_yaxis(); axes[0].set_xlabel('Vina score (kcal/mol), lower = better')
axes[0].set_title('Docking score in ABL1 (1IEP)')
axes[1].scatter(panel['heavy atoms'], panel['Vina score'], c=[colors[c] for c in panel['class']], s=80)
for _, r in panel.iterrows():
    axes[1].annotate(r.ligand, (r['heavy atoms'], r['Vina score']), fontsize=8, xytext=(3, 3), textcoords='offset points')
axes[1].set_xlabel('heavy atoms'); axes[1].set_ylabel('Vina score'); axes[1].set_title('Score vs ligand size')
for label, c in colors.items():
    axes[0].bar(0, 0, color=c, label=label)
axes[0].legend(fontsize=8, loc='lower left')
plt.tight_layout(); plt.show()

**Questions**

* Vina scores usually fall in a narrow range and become better (more negative) for **larger** molecules. Is that what you see? Compare the **ligand efficiency** (score per heavy atom) instead.
* Does every ABL1 inhibitor make the hinge H-bond? A pose of an ATP-site kinase inhibitor *without* a hinge H-bond is suspicious.
* Asciminib gets a score too — docking only looks where you tell it to (the box). It cannot know that asciminib binds elsewhere.
* Visualize a few poses with `show_pose(f'{PDBCODE}_prepped.pdb', 'dasatinib_docked.pdb')`.

In [ ]:
# Your code here, e.g. show_pose(f'{PDBCODE}_prepped.pdb', 'dasatinib_docked.pdb')

## 6. The T315I gatekeeper mutation

In CML patients treated with imatinib, the most common resistance mutation is **T315I**: the small threonine gatekeeper becomes the larger isoleucine. This removes the H-bond from the Thr OH to imatinib and makes the pocket narrower. **Ponatinib** was designed to overcome this; its rigid triple bond (alkyne) passes the isoleucine.

We build the mutant ourselves: the threonine side chain (CB, OG1, CG2) becomes isoleucine (CB, CG1, CG2, CD1) by turning OG1 into a carbon and adding one extra carbon (CD1) with standard bond lengths and angles. LePro then adds the hydrogens.

In [ ]:
def place_atom(a, b, c, bond, angle, torsion):
    """Position of atom d such that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    angle, torsion = math.radians(angle), math.radians(torsion)
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc); n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    d = np.array([-bond * math.cos(angle), bond * math.sin(angle) * math.cos(torsion), bond * math.sin(angle) * math.sin(torsion)])
    return c + d[0] * bc + d[1] * m + d[2] * n

def mutate_thr_to_ile(pdb_in, pdb_out, resid, chi2=170.0):
    lines = open(pdb_in).read().splitlines()
    res = {l[12:16].strip(): (i, np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])]))
           for i, l in enumerate(lines) if l.startswith('ATOM') and int(l[22:26]) == resid}
    assert lines[res['CA'][0]][17:20] == 'THR', f'residue {resid} is not a threonine'
    ca, cb, og1 = res['CA'][1], res['CB'][1], res['OG1'][1]
    cg1 = cb + (og1 - cb) / np.linalg.norm(og1 - cb) * 1.53          # C-C is longer than C-O
    cd1 = place_atom(ca, cb, cg1, 1.53, 113.8, chi2)

    def fmt(template, name, xyz, element):
        return f'{template[:12]}{name:>4s}{template[16:17]}ILE{template[20:30]}{xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}{template[54:76]:<22s}{element:>2s}'

    out = []
    for i, l in enumerate(lines):
        if l.startswith('ATOM') and int(l[22:26]) == resid:
            name = l[12:16].strip()
            if name == 'OG1':
                out.append(fmt(l, ' CG1', cg1, 'C'))
                out.append(fmt(l, ' CD1', cd1, 'C'))
            elif name.startswith('H') or name == 'OXT':
                continue
            else:
                out.append(l[:17] + 'ILE' + l[20:])
        else:
            out.append(l)
    open(pdb_out, 'w').write('\n'.join(out) + '\n')
    return pdb_out

GK = RES['gatekeeper']
mutate_thr_to_ile(f'protein-{PDBCODE}.pdb', f'protein-{PDBCODE}_T{GK}I.pdb', GK)
lepro(f'protein-{PDBCODE}_T{GK}I.pdb', f'{PDBCODE}_T{GK}I_prepped.pdb')
receptor_to_pdbqt(f'{PDBCODE}_T{GK}I_prepped.pdb', 'protein_T315I.pdbqt')
print(read_pdb_atoms(f'{PDBCODE}_T{GK}I_prepped.pdb').query('resid == @GK and element != "H"')[['resname', 'resid', 'name']].T)

First a simple test: how does Vina score the **crystal pose** of imatinib in the wild type and in the mutant? `v.score()` scores the pose as it is; `v.optimize()` first lets the ligand relax locally.

In [ ]:
v_mut = make_vina('protein_T315I.pdbqt')
xtal_block = MolToPDBQTBlock(xtal_lig, flexible=True, addHs=True, computeCharges=True)
rows = []
for label, v in [('wild type', v_wt), ('T315I', v_mut)]:
    v.set_ligand_from_string(xtal_block)
    rows.append({'receptor': label, 'score crystal pose': round(v.score()[0], 2),
                 'score after local optimization': round(v.optimize()[0], 2)})
pd.DataFrame(rows)

Now we dock imatinib, nilotinib and ponatinib into the mutant and compare with the wild type:

In [ ]:
rows = []
for name in ['imatinib', 'nilotinib', 'ponatinib']:
    smi = PANEL[name][0]
    mut_poses, mut_E = dock(v_mut, smi, f'{name}_T315I')
    wt_E = float(panel.loc[panel.ligand == name, 'Vina score'].iloc[0])
    rows.append({'ligand': name, 'WT score': wt_E, 'T315I score': round(mut_E[0], 2),
                 'Δ (T315I − WT)': round(mut_E[0] - wt_E, 2),
                 'pose shift WT→T315I (Å)': round(rdMolAlign.CalcRMS(mut_poses, panel_poses[name], prbId=0, refId=0), 2)})
pd.DataFrame(rows)

In [ ]:
show_pose(f'{PDBCODE}_T{GK}I_prepped.pdb', 'imatinib_T315I_docked.pdb')
display(polar_contacts(Chem.MolFromPDBFile('imatinib_T315I_docked.pdb'), -1, f'{PDBCODE}_T{GK}I_prepped.pdb', HB_RESIDUES))

**Be critical!** We expect imatinib and nilotinib to lose more score than ponatinib. With a rigid receptor, a hand-built side chain and an empirical scoring function the differences are often small (≤ 1 kcal/mol) and within the noise of the method. Try a different Ile rotamer (`mutate_thr_to_ile(..., chi2=-60)`) and see whether your conclusion changes. Quantitative predictions of resistance need more expensive methods, such as molecular dynamics and free-energy calculations.

## 7. Analyzing the results

* Does the re-docked imatinib align well with the crystal pose? If not, re-run with a different seed or a higher exhaustiveness — do you get a different result, both in predicted affinity and binding mode?
* Do the Vina scores of the panel agree with which drugs actually inhibit ABL1? How do they compare with the predictions of your QSAR / proteochemometrics model?
* Which interactions seem to matter most for binding to ABL1? Which of them are lost in the T315I mutant?
* What could improve the results? Think of: using several receptor structures (e.g. a DFG-in structure such as 2GQG for dasatinib), protonating the ligands (imatinib's piperazine is charged at pH 7.4), flexible side chains, rescoring, or molecular dynamics.